# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary information from the metadata object directly (no subscripting)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\nPublished: {meta.datePublished}\nVersion: {meta.version}\nAuthors: {[a['@id'] for a in getattr(meta, 'author', [])]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets define data tables as `record sets`. We enumerate available record sets, the fields (columns) associated with each, and their `@id`s.

In [ ]:
# List all RecordSets and their fields by @id

record_sets = dataset.metadata.recordSets
print(f"RecordSets (Data tables) found: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"RecordSet: {record_set.name}")
    print(f"  @id: {record_set['@id']}")
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {field['@id']}: {field.name} ({getattr(field, 'dataType', 'Unknown')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each RecordSet into pandas DataFrames

# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print(f"Loading RecordSets: {record_set_ids}\n")

dataframes = {}
for rs_id in record_set_ids:
    # Load records for this record set by @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"@id={rs_id}: Shape = {df.shape}")
    if len(df.columns) > 0:
        print(f"  Columns: {df.columns.tolist()[:6]}{'...' if len(df.columns)>6 else ''}")
    print()

# For demonstration, pick the first record set as primary for further exploration
primary_record_set_id = record_set_ids[0]
df = dataframes[primary_record_set_id]
print(f"Columns in selected RecordSet ({primary_record_set_id}):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. This section demonstrates:
- Filtering records using a numeric field
- Normalizing a numeric column
- Grouping by a categorical attribute

Fields and columns are referenced by their `@id`.

In [ ]:
# Choose an informative numeric field and a group field by their column names (@id from fields in RecordSet)

# You may need to customize these field ids depending on what fields are available in your data. Let's print field choices:
print("Available columns:")
print(df.columns.tolist())

# Suppose the field '@id' for peptide count is 'Peptide_count' and for abundance is 'Abundance'.
# The actual field @ids can be confirmed in section 2 above. Update as per your metadata!

numeric_field_id = None
group_field_id = None
for c in df.columns:
    if c.lower().startswith('peptide') or c.lower().startswith('abundance'):
        numeric_field_id = c
    if c.lower().startswith('description') or c.lower().startswith('category'):
        group_field_id = c

# As an example, if not found, fallback to any float or int column
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if group_field_id is None and len(df.columns) > 1:
    group_field_id = df.columns[1]

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Drop NA for robust analysis
eda_df = df.copy()
eda_df = eda_df.dropna(subset=[numeric_field_id])

# Thresholding: keep where numeric_field > threshold (e.g., median value)
threshold = eda_df[numeric_field_id].median() if numeric_field_id else 0
filtered_df = eda_df[eda_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (N={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by a field if available
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
    print(f"Grouped statistics by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True, bins=30, color='teal')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group (if grouping field exists)
if group_field_id in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we accessed and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) FAIR dataset using the `mlcroissant` library. We demonstrated how to:
- Load dataset metadata and enumerate record sets and their field `@id`s
- Extract data using record set `@id`s and load them for analysis
- Perform basic EDA including filtering, normalization, and grouping using field `@id`s
- Visualize metrics and distributions for numeric fields

For deeper scientific questions, explore the field documentation and available field `@id`s through the metadata, and tailor the above workflow to your use case.